# PolyClustX: LMGEC experiments, Optuna search, and visualization

This reference notebook discovers the available models and datasets, compares **three LMGEC configurations on two datasets**, tunes LMGEC with Optuna, and visualizes the results.

Run the cells in order in a Python kernel with PolyClustX installed. If needed, install the package and Jupyter from a terminal:

```bash
python -m pip install polyclustx notebook
jupyter notebook
```

The notebook is intentionally saved without outputs. Dataset files are downloaded on first use and cached for later runs. Reports and figures are saved under `experiments/lmgec-reference/` relative to the notebook's working directory.

LMGEC fitting is unsupervised. Reference labels are used for benchmark evaluation and NMI-based hyperparameter selection, not supplied to LMGEC.

In [ ]:
import polyclustx

print("PolyClustX version:", polyclustx.__version__)

## 1. Import LMGEC and discover the catalog

`list_models()` lists the methods registered in the framework; `list_datasets()` lists the dataset families. `load_data()` loads a dataset by its catalog name.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from polyclustx.models import LMGEC, list_models
from polyclustx.data import load_data, list_datasets
from polyclustx.experiment import Experiment, OptunaSearchExperiment
from polyclustx.metrics import clustering_accuracy, nmi_score, ari_score

print("Available methods:")
print(", ".join(list_models()))

print("\nAvailable datasets:")
print(", ".join(list_datasets()))

## 2. Load two datasets

**ACM** contains academic papers with keyword features and paper relationships. **IMDB** contains movies with plot features and actor/director relationships.

Both benchmarks have **three reference classes**, so the same model configurations can be evaluated on both datasets in one `Experiment`. The default loader constructs their views from the stored features and graphs. LMGEC handles its own normalization and graph propagation.

Loading requires internet access only when a file is not already cached.

In [ ]:
datasets = {
    "ACM": load_data("acm"),
    "IMDB": load_data("imdb"),
}

display(pd.DataFrame([
    {
        "Dataset": name,
        "Samples": data.n_samples,
        "Views": data.n_views,
        "Feature dimensions": data.feature_dims,
        "Reference classes": data.n_clusters,
        "Has graphs": data.has_graphs,
        "Complete": data.is_complete,
    }
    for name, data in datasets.items()
]).set_index("Dataset"))

## 3. Prepare three LMGEC configurations

Compare embedding dimensions **8, 16, and 32**, with other model settings held fixed. Give each configuration a readable name.

Each configuration will be fitted **three times per dataset**: 2 datasets × 3 configurations × 3 runs = **18 fits**. The runner uses seeds 42, 43, and 44 for the independent repetitions. CPU execution makes this example portable; change `device="cpu"` to `device="auto"` to allow GPU execution.

If you replace these datasets with benchmarks having different class counts, prepare separate experiments with each dataset's own `n_clusters`.

In [ ]:
seed = 42
runs = 3
max_iter = 30
output_dir = Path("experiments") / "lmgec-reference"
output_dir.mkdir(parents=True, exist_ok=True)

n_clusters = datasets["ACM"].n_clusters
assert all(data.n_clusters == n_clusters for data in datasets.values()), (
    "This shared comparison needs the same K for both datasets. "
    "For different class counts, prepare a separate experiment per dataset."
)

models = [
    LMGEC(n_clusters=n_clusters, embedding_dim=8, temperature=1.0,
          max_iter=max_iter, device="cpu"),
    LMGEC(n_clusters=n_clusters, embedding_dim=16, temperature=1.0,
          max_iter=max_iter, device="cpu"),
    LMGEC(n_clusters=n_clusters, embedding_dim=32, temperature=1.0,
          max_iter=max_iter, device="cpu"),
]
for model, name in zip(models, ["LMGEC-8d", "LMGEC-16d", "LMGEC-32d"]):
    model.name = name

metrics = {
    "ACC": clustering_accuracy,
    "NMI": nmi_score,
    "ARI": ari_score,
}

## 4. Run the comparison and report the results

`Experiment.run()` prints a **hyperparameter table** and a **results table for each dataset**, reporting mean ± standard deviation and elapsed time. A star marks the best mean for each metric. `verbose=2` enables progress; use `verbose=1` for tables only.

The returned result contains per-run scores and loss histories. Its JSON report is saved automatically; no scores are prefilled in this notebook.

In [ ]:
results = Experiment(
    datasets=datasets,
    models=models,
    metrics=metrics,
    runs=runs,
    seed=seed,
    benchmark=True,
    verbose=2,
    experiment_name="LMGEC comparison on ACM and IMDB",
    save_path=str(output_dir),
).run()

print("Comparison JSON:", results.json_path)

## 5. Tune LMGEC on ACM with Optuna

Search categorical choices for `embedding_dim` and `beta`, and a continuous logarithmic range for `temperature`. Keep K, iteration budget, and device fixed.

The objective is **mean NMI over two independent runs** per trial. This example uses **12 trials** (24 fits); increase the budget for a more extensive search. Optuna's TPE sampler starts with exploratory trials before using earlier scores to guide subsequent suggestions.

NMI-based selection uses benchmark labels. The selected scores are tuning results, not an independent held-out performance estimate. For unlabeled data, select an internal metric such as Silhouette instead.

In [ ]:
search_data = datasets["ACM"]

search = OptunaSearchExperiment(
    model_class=LMGEC,
    param_distributions={
        "n_clusters": [search_data.n_clusters],
        "embedding_dim": [8, 16, 32],
        "temperature": {"low": 0.1, "high": 10.0, "log": True},
        "beta": [0.5, 1.0, 2.0],
        "max_iter": [max_iter],
        "device": ["cpu"],
    },
    dataset=search_data,
    metrics=metrics,
    best_metric="NMI",
    n_iter=12,
    runs=2,
    seed=seed,
    benchmark=True,
    verbose=2,
    experiment_name="Optuna LMGEC tuning on ACM",
    save_path=str(output_dir),
)
search_results = search.run()

print("Search JSON:", search_results.json_path)

### Inspect the winning configuration

The search prints its trial table automatically. Below, inspect the selected parameters and their repeated-run mean and standard deviation. The JSON also records every trial and the best-so-far tuning steps.

In [ ]:
if search_results.best_trial is None:
    raise RuntimeError("No trial produced a valid objective. Inspect the search report before continuing.")

print("Best hyperparameters:")
display(pd.Series(search_results.best_params, name="Value").to_frame())

best_entry = search_results.to_dict()["experiments"][search_results.best_idx]
print("Winning configuration: repeated-run scores")
display(pd.DataFrame(best_entry["aggregate"]["metrics"]).T[["mean", "std", "count"]])

print(f"Selected mean NMI: {search_results.best_score:.4f}")

## 6. Visualize the experiment results

Use PolyClustX's plotting functions, not a separate plotting pipeline. The first plot compares the normalized loss trajectories from the **first run of each of the six dataset/configuration pairs**.

Normalization compares convergence shapes, not absolute objective values. Set `run=None` in `plot_loss_comparison` to display all repetitions.

In [ ]:
from polyclustx.visualization import plot_clusters, plot_loss, plot_loss_comparison

comparison_figure, comparison_axis = plt.subplots(figsize=(12, 5))
plot_loss_comparison(results, run=1, ax=comparison_axis, show=False)
comparison_figure.tight_layout()
plt.show()

### Inspect the tuned model's embedding

Experiment runners fit fresh model copies and return reports, not fitted estimators. Refit the winning configuration with seed 42 to inspect a **representative first run**. This is one additional fit, not the average over the search repetitions.

The fitted model exposes `labels_`, `embedding_`, `centroids_`, and `loss_history_`. Reference and predicted labels color the **same PCA projection** in the next cell; arbitrary cluster IDs need not match reference class IDs.

In [ ]:
best_model = LMGEC(**search_results.best_params, random_state=seed)
search_data.fit(best_model)

print("Embedding shape:", best_model.embedding_.shape)
print("Centroid shape:", best_model.centroids_.shape)
display(pd.Series(
    {name: metric(search_data.labels, best_model.labels_) for name, metric in metrics.items()},
    name="Representative run score",
).to_frame())

In [ ]:
best_figure, axes = plt.subplots(1, 3, figsize=(17, 4.5))

plot_clusters(
    best_model.embedding_, labels=search_data.labels, method="pca",
    random_state=seed, ax=axes[0], show=False, title="ACM: reference classes",
)
plot_clusters(
    best_model.embedding_, labels=best_model.labels_, method="pca",
    random_state=seed, ax=axes[1], show=False, title="ACM: predicted clusters",
)
plot_loss(
    best_model.loss_history_, normalize=True, ax=axes[2],
    show=False, title="Tuned LMGEC: normalized loss",
)

best_figure.tight_layout()
plt.show()

## 7. Save tables, reports, and figures

The comparison and search JSON files were saved by their runners. Export CSV tables for analysis and ZIP bundles containing each report plus its CSV table. Save the figures separately as SVG files.

Output paths are relative to the kernel's working directory; the final printed path shows where to find them.

In [ ]:
results.export(output_dir / "lmgec-comparison.csv")
search_results.export(output_dir / "lmgec-optuna.csv")
results.export(output_dir / "lmgec-comparison.zip")
search_results.export(output_dir / "lmgec-optuna.zip")

figure_dir = output_dir / "figures"
figure_dir.mkdir(exist_ok=True)
comparison_figure.savefig(figure_dir / "comparison-losses.svg", bbox_inches="tight")
best_figure.savefig(figure_dir / "tuned-lmgec.svg", bbox_inches="tight")

print("Saved reports, tables, and figures in:", output_dir.resolve())

To show the comparison tables again without fitting, use `print(results.results_tables())`. To inspect all Optuna trial objects, use `search.study_.trials`.

To try another registered method, import it from `polyclustx.models` and replace the estimator configurations and search parameters. See the [model guide](https://polyclustx.readthedocs.io/en/latest/user_guide/clustering_models.html) for supported inputs and the [experiment guide](https://polyclustx.readthedocs.io/en/latest/user_guide/experiments.html) for metrics and report formats.